# EPI Vision — retreino (fine-tuning) do YOLO

Continua a partir do `best.pt` atual (NÃO treina do zero) e acrescenta variedade de cor.

**Cores a cobrir (pesquisa da Sophia sobre o Porto do Itaqui; confirmar com o porto):**
- Coletes: verde neon, laranja, azul escuro
- Capacetes: branco, azul escuro, vermelho, laranja, amarelo, verde

**Antes de rodar:** Ambiente de execução → Alterar tipo → **GPU (T4)**.

Regra de ouro: misture os dados NOVOS com os ANTIGOS. Treinar só com os novos faz o modelo esquecer o resto.
Classes mantidas iguais às do modelo atual (10 classes, Construction-Site-Safety).

In [ ]:
!nvidia-smi | head -15
!pip -q install ultralytics roboflow

## 1. Enviar o best.pt atual
Use o botão de upload à esquerda (pasta) e envie o `best.pt`, ou rode a célula abaixo.

In [ ]:
from google.colab import files
up = files.upload()            # escolha o best.pt
BEST = list(up.keys())[0]
print('modelo base:', BEST)

## 2. Dataset ANTIGO (o mesmo do treino original)
Preencha com a sua chave do Roboflow (Settings → API Key). Não compartilhe a chave.
Workspace/projeto/versão: use os mesmos do treino original (Construction-Site-Safety, versão 27).

In [ ]:
from roboflow import Roboflow
API_KEY   = 'COLE_SUA_CHAVE_AQUI'
WORKSPACE = 'PREENCHA'     # igual ao do treino original
PROJECT   = 'PREENCHA'     # Construction-Site-Safety
VERSION   = 27
rf = Roboflow(api_key=API_KEY)
old = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8', location='/content/old')
print(old.location)

## 3. Dataset NOVO (coletes e capacetes de várias cores)
Duas fontes, usadas juntas:
1. **Fotos do porto / de EPIs reais** que você enviar, anotadas no Roboflow (use o modelo atual para pré-anotar e só corrija).
2. **Datasets públicos do Roboflow Universe** (busque: `safety vest blue`, `hard hat colors`, `helmet vest detection`). Confira a licença antes de usar.

**Importante:** as classes do dataset novo precisam ter os MESMOS nomes do modelo atual:
`Hardhat`, `Safety Vest`, `Person` (e as demais). No Roboflow, use *Modify Classes* para renomear antes de exportar.

In [ ]:
# (workspace, projeto, versão, mapa: classe do dataset -> nome da classe do NOSSO modelo)
NEW_SOURCES = [
    # college-2t4kc/safety-vest-detection (CC BY 4.0, autor "college"): 0 = capacete, 1 = colete
    ('college-2t4kc', 'safety-vest-detection', 1, {0: 'Hardhat', 1: 'Safety Vest'}),
]
new_dirs, new_maps = [], []
for i, (ws, pj, v, cmap) in enumerate(NEW_SOURCES):
    d = rf.workspace(ws).project(pj).version(v).download('yolov8', location=f'/content/new{i}')
    new_dirs.append(d.location); new_maps.append(cmap)
print(new_dirs)

## 4. Conferir o mapeamento de classes (automático)
Roda o `best.pt` atual em até 120 imagens de cada dataset novo e vê, para cada classe do dataset novo,
qual classe o nosso modelo enxerga nas mesmas caixas. Se o mapa escolhido não bater, o notebook PARA com aviso.

In [ ]:
import yaml, glob, os
from ultralytics import YOLO

def names(path):
    n = yaml.safe_load(open(path + '/data.yaml'))['names']
    return list(n.values()) if isinstance(n, dict) else list(n)

base = names(old.location)
print('classes base:', base)
low = [n.lower() for n in base]
def cid(name):
    assert name.lower() in low, f'classe {name!r} não existe no dataset base: {base}'
    return low.index(name.lower())

det = YOLO(BEST)

def read_gt(p):
    if not os.path.exists(p): return []
    out = []
    for l in open(p).read().splitlines():
        r = l.split()
        if len(r) >= 5: out.append((int(float(r[0])), *map(float, r[1:5])))
    return out

def iou(a, b):
    ax1, ay1, ax2, ay2 = a[0]-a[2]/2, a[1]-a[3]/2, a[0]+a[2]/2, a[1]+a[3]/2
    bx1, by1, bx2, by2 = b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
    iw, ih = max(0, min(ax2, bx2)-max(ax1, bx1)), max(0, min(ay2, by2)-max(ay1, by1))
    inter = iw*ih
    return inter / (a[2]*a[3] + b[2]*b[3] - inter + 1e-9)

for d, cmap in zip(new_dirs, new_maps):
    imgs = sorted(glob.glob(f'{d}/train/images/*'))[:120]
    votes = {}
    for im in imgs:
        gt = read_gt(im.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt')
        r = det.predict(im, conf=0.25, verbose=False)[0]
        preds = list(zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist()))
        for c, *box in gt:
            best = max(((iou(box, pb), int(pc)) for pc, pb in preds), default=(0, None))
            if best[0] > 0.5:
                votes.setdefault(c, {}).setdefault(base[best[1]], 0)
                votes[c][base[best[1]]] += 1
    print(d, '-> votos (classe do dataset novo: classes que o nosso modelo viu):')
    for c, v in sorted(votes.items()):
        print('  ', c, dict(sorted(v.items(), key=lambda x: -x[1])))
        top = max(v, key=v.get)
        if c in cmap and sum(v.values()) >= 10:
            assert top.lower() == cmap[c].lower(), (
                f'ATENÇÃO: classe {c} parece ser {top!r}, mas o mapa diz {cmap[c]!r}. Corrija NEW_SOURCES.')
    unmapped = [c for c in votes if c not in cmap]
    print('  classes sem mapa (serão ignoradas):', unmapped)
print('mapeamento conferido.')

## 5. Juntar tudo em um único dataset
- Dataset antigo: copiado como está.
- Datasets novos: rótulos convertidos para os NOSSOS ids de classe, e as **pessoas são marcadas automaticamente**
  pelo `best.pt` atual (confiança ≥ 0,60), porque esses datasets não têm a classe Person e, sem isso, o modelo
  aprenderia que "pessoa sem rótulo = fundo". Confira uma amostra visualmente no passo 5b.

In [ ]:
import shutil, os, yaml
OUT = '/content/merged'
for split in ('train', 'valid', 'test'):
    for kind in ('images', 'labels'):
        os.makedirs(f'{OUT}/{split}/{kind}', exist_ok=True)

def copy_old(src, tag):
    for split in ('train', 'valid', 'test'):
        for kind in ('images', 'labels'):
            s = f'{src}/{split}/{kind}'
            if os.path.isdir(s):
                for f in os.listdir(s):
                    shutil.copy(f'{s}/{f}', f'{OUT}/{split}/{kind}/{tag}_{f}')

person_id = cid('Person')

def copy_new(src, cmap, tag):
    n_p = 0
    for split in ('train', 'valid', 'test'):
        simg, slab = f'{src}/{split}/images', f'{src}/{split}/labels'
        if not os.path.isdir(simg):
            continue
        for f in os.listdir(simg):
            stem = os.path.splitext(f)[0]
            lines = []
            lp = f'{slab}/{stem}.txt'
            if os.path.exists(lp):
                for l in open(lp).read().splitlines():
                    r = l.split()
                    if len(r) >= 5 and int(float(r[0])) in cmap:
                        lines.append(' '.join([str(cid(cmap[int(float(r[0]))]))] + r[1:5]))
            res = det.predict(f'{simg}/{f}', conf=0.60, classes=[person_id], verbose=False)[0]
            for b in res.boxes.xywhn.tolist():
                lines.append(f'{person_id} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}')
                n_p += 1
            shutil.copy(f'{simg}/{f}', f'{OUT}/{split}/images/{tag}_{f}')
            open(f'{OUT}/{split}/labels/{tag}_{stem}.txt', 'w').write('\n'.join(lines))
    print(tag, '-> pessoas marcadas automaticamente:', n_p)

copy_old(old.location, 'old')
for i, (d, cmap) in enumerate(zip(new_dirs, new_maps)):
    copy_new(d, cmap, f'new{i}')

yaml.safe_dump({'path': OUT, 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
                'names': base}, open(f'{OUT}/data.yaml', 'w'))
for s in ('train', 'valid', 'test'):
    print(s, len(os.listdir(f'{OUT}/{s}/images')), 'imagens')

### 5b. Conferir uma amostra do dataset misturado
Mostra 6 imagens NOVAS já com as caixas convertidas (capacete, colete e pessoa). Se algo estiver trocado ou faltando, não siga para o treino.

In [ ]:
import random, cv2, glob
from IPython.display import display
from PIL import Image
imgs = [p for p in glob.glob(f'{OUT}/train/images/new*')]
random.seed(1)
for p in random.sample(imgs, min(6, len(imgs))):
    im = cv2.imread(p); h, w = im.shape[:2]
    lab = open(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt').read().splitlines()
    for l in lab:
        c, x, y, bw, bh = l.split(); c = int(c); x, y, bw, bh = [float(v) for v in (x, y, bw, bh)]
        p1, p2 = (int((x-bw/2)*w), int((y-bh/2)*h)), (int((x+bw/2)*w), int((y+bh/2)*h))
        cv2.rectangle(im, p1, p2, (0, 255, 0), 2); cv2.putText(im, base[c], p1, 0, 0.6, (0, 255, 0), 2)
    display(Image.fromarray(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)).resize((480, int(480*h/w))))

## 6. Treinar (fine-tuning)
- Parte do `best.pt`; menos épocas que o treino original.
- `hsv_h=0.5` + `hsv_s`: aumento forte de variação de cor, para o modelo depender menos da cor do colete/capacete e mais da forma e das faixas. **É um experimento**: compare com a execução padrão (hsv_h=0.015) e fique com a que tiver melhor resultado no conjunto de validação E nas fotos reais do porto.
- `patience` interrompe sozinho se parar de melhorar.

In [ ]:
from ultralytics import YOLO
model = YOLO(BEST)
res = model.train(data='/content/merged/data.yaml', epochs=60, imgsz=640, batch=16, patience=15,
                  hsv_h=0.5, hsv_s=0.7, hsv_v=0.4, project='/content/runs', name='finetune_cores', device=0)

## 7. Avaliar por classe (compare com o modelo antigo: Hardhat 0,804 / Safety Vest 0,791 / Person 0,758 de mAP50)

In [ ]:
new = YOLO('/content/runs/finetune_cores/weights/best.pt')
m = new.val(data='/content/merged/data.yaml')
for i, n in enumerate(m.names.values()):
    print(f'{n:15s} mAP50={m.box.ap50[i]:.3f}')
print('mAP50 geral:', round(m.box.map50, 3))

## 8. Testar nas SUAS imagens (o teste que realmente importa)
Envie fotos de coletes azul-escuro, verde neon e laranja, e de capacetes das 6 cores, **que não foram usadas no treino**.
O que olhar: o colete azul-escuro agora aparece como `Safety Vest`? A regata azul NÃO deve virar colete.

In [ ]:
up = files.upload()
for name in up:
    r = new.predict(name, conf=0.25, save=True, project='/content/pred', name='teste', exist_ok=True)
    print(name, [(new.names[int(c)], round(float(s), 2)) for c, s in zip(r[0].boxes.cls, r[0].boxes.conf)])
from IPython.display import Image, display
for f in sorted(glob.glob('/content/pred/teste/*')):
    display(Image(f, width=480))

## 9. Baixar o novo modelo
Guarde o `best.pt` antigo; só troque depois de testar na câmera (v14 + novo modelo).

In [ ]:
files.download('/content/runs/finetune_cores/weights/best.pt')